In [8]:
import boto3
import os


ENDPOINT = 'https://s3.us-west-002.backblazeb2.com'
KEY_ID = '0050848015934060000000004' 
APPLICATION_KEY = 'K0053yRolTcPrQsZ08oR0kAQc35BWo0' 
BUCKET_NAME = 'qdc-data-003'

# Local path where the bucket contents will be saved
LOCAL_DOWNLOAD_DIR = 'qdc-data-003_download'

# --- Initialization ---
session = boto3.session.Session()

s3 = session.resource(
    service_name='s3',
    endpoint_url=ENDPOINT,
    aws_access_key_id=KEY_ID,
    aws_secret_access_key=APPLICATION_KEY,
    region_name='us-west-002'
)
bucket = s3.Bucket(BUCKET_NAME)

# Create the local download directory if it does not exist
if not os.path.exists(LOCAL_DOWNLOAD_DIR):
    os.makedirs(LOCAL_DOWNLOAD_DIR)
    print(f"Created local directory: {LOCAL_DOWNLOAD_DIR}")

# --- Download Logic ---
print(f"Starting download from bucket '{BUCKET_NAME}'...")

downloaded_count = 0
for obj in bucket.objects.all():
    # obj.key is the file path within the bucket (e.g., 'data/file1.csv')
    remote_path = obj.key
    
    # Construct the full local path for the file
    local_path = os.path.join(LOCAL_DOWNLOAD_DIR, remote_path)
    
    # Ensure the local subdirectory structure exists
    local_dir = os.path.dirname(local_path)
    if not os.path.exists(local_dir):
        os.makedirs(local_dir)
    
    try:
        # Download the file
        bucket.download_file(remote_path, local_path)
        downloaded_count += 1
        print(f"Downloaded: {remote_path}")
    except Exception as e:
        # Log any files that fail to download
        print(f"Error downloading {remote_path}: {e}")

print("---")
print(f"Download complete. Total files downloaded: {downloaded_count}")

Starting download from bucket 'qdc-data-003'...


ClientError: An error occurred (InvalidAccessKeyId) when calling the ListObjects operation: The key '0050848015934060000000004' is not valid

In [9]:
pip install b2sdk

  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)

   ----- ---------------------------------- 1/7 [idna]
   ----------- ---------------------------- 2/7 [charset_normalizer]
   ---------------------------- ----------- 5/7 [requests]
   ---------------------------- ----------- 5/7 [requests]
   ---------------------------------- ----- 6/7 [b2sdk]
   ---------------------------------- ----- 6/7 [b2sdk]
   ---------------------------------- ----- 6/7 [b2sdk]
   ---------------------------------- ----- 6/7 [b2sdk]
   ---------------------------------- ----- 6/7 [b2sdk]
   ---------------------------------- ----- 6/7 [b2sdk]
   ---------------------------------- ----- 6/7 [b2sdk]
   ---------------------------------- ----- 6/7 [b2sdk]
   ---------------------------------- ----- 6/7 [b2sdk]
   ---------------------------------- ----- 6/7 [b2sdk]
   ---------------------------------- ----- 6/7 [b2sdk]
   -------


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import os
from b2sdk.v2 import B2Api, InMemoryAccountInfo

# --- Configuration Parameters ---
# Your provided credentials
KEY_ID = '0050848015934060000000004' # B2 Application Key ID
APPLICATION_KEY = 'K0053yRolTcPrQsZ08oR0kAQc35BWo0' # B2 Application Key
BUCKET_NAME = 'qdc-data-003'

# Local path where the bucket contents will be saved
LOCAL_DOWNLOAD_DIR = 'qdc-data-003_download'

# --- Initialization ---
info = InMemoryAccountInfo()
b2_api = B2Api(info)

# The b2sdk authorize_account method handles the endpoint automatically
print("Authorizing account...")
try:
    b2_api.authorize_account('production', KEY_ID, APPLICATION_KEY)
except Exception as e:
    print(f"Authorization failed: {e}")
    # This key validation failure (ClientError) from boto3 may still occur here
    # if the key is deleted or its permissions are wrong, but is less common.
    exit(1)

# Get the bucket object
try:
    bucket = b2_api.get_bucket_by_name(BUCKET_NAME)
except Exception as e:
    print(f"Error accessing bucket '{BUCKET_NAME}': {e}")
    exit(1)

# Create the local download directory if it does not exist
if not os.path.exists(LOCAL_DOWNLOAD_DIR):
    os.makedirs(LOCAL_DOWNLOAD_DIR)
    print(f"Created local directory: {LOCAL_DOWNLOAD_DIR}")

# --- Download Logic ---
print(f"Starting download from bucket '{BUCKET_NAME}'...")

downloaded_count = 0
# bucket.ls() lists file versions and preserves structure
for file_version, folder_name in bucket.ls(show_versions=False):
    # Skip directories or empty folders that might be returned
    if file_version.file_name is None:
        continue
    
    remote_path = file_version.file_name
    local_path = os.path.join(LOCAL_DOWNLOAD_DIR, remote_path)
    
    # Ensure the local subdirectory structure exists
    local_dir = os.path.dirname(local_path)
    if not os.path.exists(local_dir):
        os.makedirs(local_dir)
    
    try:
        # download_file_by_name is a simpler download method
        bucket.download_file_by_name(
            file_name=remote_path,
            download_dest=local_path
        )
        downloaded_count += 1
        print(f"Downloaded: {remote_path}")
    except Exception as e:
        print(f"Error downloading {remote_path}: {e}")

print("---")
print(f"Download complete. Total files downloaded: {downloaded_count}")

Authorizing account...
Starting download from bucket 'qdc-data-003'...


TypeError: ls() got an unexpected keyword argument 'show_versions'